# MSD（Decathlon）分割微调 + AR-SSL 预训练 checkpoint

**说明**：沿用 `downstream/segmentation/main.py` 单卡流程（不要使用 `--distributed`）。请**按顺序**运行：**① PyTorch（cu130，含 torch / torchvision / torchaudio）**，**② CUDA / 自检**，再 **③** 其余依赖。**Restart Kernel 后 Jupyter 变量会清空，`import torch` 会失败**，请务必**从顶部重新执行**：先 **①**，再跑后面（不要跳过 ①）。

**安装必须与当前内核共用同一解释器**：下面代码格用 **`sys.executable -m pip`**（不用 `%pip`），避免装进别的 Python 或与 **`%AppData%\\Roaming\\Python`** 混用。

**环境提示（Python 3.13）**：`cu130` 索引上 **Windows + cp313** 的 GPU 包从约 **torch 2.9** 起才有；若环境里已是 **`x.x.x+cpu`**，必须 **卸载后强制重装** 才会换成 **`+cu130`**。若 pip 出现 **“Defaulting to user installation…”**，说明包进了 **`%AppData%\Roaming\Python`**，易与 Miniconda 混用；建议用**可写的 conda 环境**（或修好 `ProgramData\miniconda3` 目录写权限），避免只装在 Roaming 里。

原始论文分割下游常用 **MSD Task06（Lung）**。本机数据若在二级目录下，例如 `...\Task06_Lung-001\Task06_Lung`，请把 **`MSD_TASK_DIR`** 指到最内层 **`Task06_Lung`**（与 `dataset.json` 同级）。

## 期望目录结构（Task06_Lung）

将 **`MSD_TASK_DIR`** 指到任务根目录（与 `imagesTr` / `labelsTr` / `dataset.json` 同级）。若仅有官方 **`dataset.json`**（无 `validation`），会在**下面的配置单元**里**自动生成** `dataset_withVal_autogen.json`（从 `training` 末尾约 20% 划为验证集）。若你已有 **`dataset_withVal.json`**，将优先使用它。

```
MSD_TASK_DIR/
  dataset.json                 # 官方常见；无 validation 时会自动生成 dataset_withVal_autogen.json
  dataset_withVal.json         # 若已有人工划分则优先使用
  imagesTr/                  # *.nii.gz
  labelsTr/                  # 与 MSD 配套的 label
  （部分任务还有 imagesTs 等）
```

JSON 中 `training` / `validation` 列表里的 `image` / `label` 路径一般为**相对 `MSD_TASK_DIR` 的路径**。

**流程**：① PyTorch（cu130）安装 → ② CUDA / 版本自检 → ③ 其余依赖 → ④ 配置路径 → ⑤ 目录检查 → ⑥ 封装 checkpoint → ⑦ 启动训练（含 CUDA）。

In [ ]:
# 用「当前 Jupyter 内核」的 Python 调用 pip（Restart 后须先重新运行本格，否则会 No module named 'torch'）
import os
import subprocess
import sys

_ENV = os.environ.copy()
_ENV.pop("PIP_USER", None)


def _pip(args: list[str]) -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip"] + args,
        env=_ENV,
    )


# PyTorch 官方 CUDA 三件套 — cu130；Python 3.13 通常为 2.9+ GPU wheel
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
    env=_ENV,
    check=False,
)
_pip(
    [
        "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "torch",
        "torchvision",
        "torchaudio",
        "--index-url",
        "https://download.pytorch.org/whl/cu130",
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-c",
        "import sys; print('解释器:', sys.executable); import torch; print('torch:', torch.__version__, '->', torch.__file__)\n",
    ],
)
import site as _site
_ug = getattr(_site, "getusersitepackages", None)
if callable(_ug):
    print("用户 site-packages:", _ug(), "——若这里的 torch 与 conda 混在一起，易出现 Restart 或混版本问题；可清空其中 torch/torchaudio/torchvision 后再跑本格。")
print("安装格执行完毕（若 Restart 请先重跑本格再跑自检）。")


Found existing installation: torch 2.6.0
Uninstalling torch-2.6.0:
  Successfully uninstalled torch-2.6.0
Found existing installation: torchvision 0.21.0
Uninstalling torchvision-0.21.0:
  Successfully uninstalled torchvision-0.21.0
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.


In [3]:
# --- 安装 GPU 版 PyTorch 后：自检 wheel 是否与 CUDA runtime 对上 ---
# pip 若有 "Restart kernel..." 或其它升级，请先 Restart，再单独运行本格。
import shutil
import subprocess
import sys

try:
    import torch
except ImportError as exc:
    raise ImportError(
        "找不到 torch。
"
        "① Restart Kernel 后必须**先重新运行上一格 PyTorch 安装**；② 若在终端用别的 python pip 装了包，会与当前内核不符。"
    ) from exc

print("torch 文件路径（应在当前解释器 env 或 site-packages）:")
print(" ", getattr(torch, "__file__", "?"))

print("===== PyTorch / CUDA 自检 =====")
print("Python:", sys.version.split()[0], "| 解释器:", sys.executable)
print("torch:", torch.__version__)
print("torch.version.cuda (wheel 编译时 CUDA):", torch.version.cuda)
print("torch.backends.cudnn.version():", torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None)

cuda_ok = torch.cuda.is_available()
print("torch.cuda.is_available():", cuda_ok)
if cuda_ok:
    nd = torch.cuda.device_count()
    print("torch.cuda.device_count():", nd)
    for i in range(nd):
        caps = torch.cuda.get_device_capability(i)
        print(
            f"  GPU{i}:",
            torch.cuda.get_device_name(i),
            "| compute capability:",
            ".".join(str(x) for x in caps),
        )
else:
    print("⚠ GPU 不可用：常见为 CPU-only 轮子或驱动/环境与 wheel 不匹配。请核对上一格是否真的来自 cu*** 索引，并检查 nvidia 驱动。")

try:
    import torchvision

    print("torchvision:", torchvision.__version__)
except ImportError:
    print("torchvision: 未导入（若需请确认与上一格一起安装成功）")

try:
    import torchaudio

    print("torchaudio:", torchaudio.__version__)
except (ImportError, OSError, RuntimeError) as _e:
    print("torchaudio: 导入失败（本分割管线可不依赖）；若介意可忽略或重装与 torch 同源的三件套——", repr(_e))

_nvidia_smi = shutil.which("nvidia-smi")
if _nvidia_smi:
    try:
        r = subprocess.run(
            [_nvidia_smi, "-L"],
            capture_output=True,
            text=True,
            timeout=15,
        )
        print("----- nvidia-smi -L -----")
        print(r.stdout.strip() or r.stderr.strip() or "(无输出)")
    except Exception as e:
        print("nvidia-smi 调用失败:", e)
else:
    print("----- nvidia-smi：未在 PATH 中找到 -----")

print("================================")


torch 文件路径（应在当前解释器 env 或 site-packages）:
  C:\Users\hrole\AppData\Roaming\Python\Python313\site-packages\torch\__init__.py
===== PyTorch / CUDA 自检 =====
Python: 3.13.2 | 解释器: c:\ProgramData\miniconda3\python.exe
torch: 2.6.0+cpu
torch.version.cuda (wheel 编译时 CUDA): None
torch.backends.cudnn.version(): None
torch.cuda.is_available(): False
⚠ GPU 不可用：常见为 CPU-only 轮子或驱动/环境与 wheel 不匹配。请核对上一格是否真的来自 cu*** 索引，并检查 nvidia 驱动。
torchvision: 0.21.0+cpu


OSError: [WinError 127] 找不到指定的程序。

In [4]:
# --- 其余分割下游依赖（不含 torch / torchvision / torchaudio；见 cu130「安装」「自检」）---
import subprocess
import sys

_DEPS = (
    "monai",
    "tensorboardX",
    "timm",
    "transformers",
    "scipy",
    "h5py",
    "batchgenerators",
    "nibabel",
    "SimpleITK",
)

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + list(_DEPS))
print("已安装/更新 pip 包:", ", ".join(_DEPS))


已安装/更新 pip 包: monai, tensorboardX, timm, transformers, scipy, h5py, batchgenerators, nibabel, SimpleITK


In [5]:
from __future__ import annotations

import json
from pathlib import Path

# ——MSD Task06 Lung：与论文一致；目录为你本机路径（已在本环境验证存在 dataset.json / imagesTr / labelsTr）
MSD_TASK_DIR = Path(r"D:\finetune\MSD\Task06_Lung-001\Task06_Lung")


def resolve_msd_json_list(task_dir: Path) -> str:
    """优先 dataset_withVal.json；否则从官方 dataset.json 自动生成带 validation 的列表（MONAI get_loader 需要 validation）。"""
    task_dir = Path(task_dir).resolve()
    pref = task_dir / "dataset_withVal.json"
    if pref.is_file():
        print("使用已有:", pref.name)
        return "dataset_withVal.json"
    autogen = task_dir / "dataset_withVal_autogen.json"
    if autogen.is_file():
        print("使用已有:", autogen.name)
        return "dataset_withVal_autogen.json"
    base = task_dir / "dataset.json"
    if not base.is_file():
        return "dataset_withVal.json"
    obj = json.loads(base.read_text(encoding="utf-8"))
    train = obj.get("training")
    if not train or len(train) < 3:
        raise ValueError("dataset.json 缺少 training 或样本过少")
    if "validation" in obj:
        print("dataset.json 已含 validation，直接使用")
        return "dataset.json"
    n_val = max(1, len(train) // 5)
    obj["validation"] = train[-n_val:]
    obj["training"] = train[:-n_val]
    obj["numValidation"] = len(obj["validation"])
    obj["numTraining"] = len(obj["training"])
    out = task_dir / "dataset_withVal_autogen.json"
    out.write_text(json.dumps(obj, indent=2), encoding="utf-8")
    print("已从 dataset.json 生成:", out.name, "（约 20% 原 training → validation）")
    return out.name


# 预训练权重（ReconModel 等，内含 model.*）
PRETRAIN_CKPT = Path(r"D:\finetune\ar-ssl4m\ar-ssl4m\checkpoints\0\0.pth")

REPO_ROOT = Path(r"D:\Cursor\AR-SSL4M-DEMO").resolve()

SAVE_BASE = Path(r"D:\finetune\msd_seg_ssl_runs")

DEMO_QUICK = True

assert MSD_TASK_DIR.is_dir(), (
    f"未见任务目录: {MSD_TASK_DIR}\n请确认路径为最内层 Task06_Lung（含 dataset.json）"
)
assert PRETRAIN_CKPT.is_file(), f"未见权重: {PRETRAIN_CKPT}"
assert REPO_ROOT.is_dir()

JSON_LIST = resolve_msd_json_list(MSD_TASK_DIR)

# main.py：data_path = join(MSD_data_base, task_name)
MSD_DATA_BASE = MSD_TASK_DIR.parent
TASK_NAME = MSD_TASK_DIR.name

SEG_DIR = REPO_ROOT / "downstream" / "segmentation"
print("MSD_TASK_DIR:", MSD_TASK_DIR)
print("MSD_DATA_BASE + TASK_NAME:", MSD_DATA_BASE, "+", TASK_NAME)
print("JSON_LIST:", JSON_LIST)
print("PRETRAIN_CKPT:", PRETRAIN_CKPT)
print("SAVE_BASE:", SAVE_BASE)


使用已有: dataset_withVal_autogen.json
MSD_TASK_DIR: D:\finetune\MSD\Task06_Lung-001\Task06_Lung
MSD_DATA_BASE + TASK_NAME: D:\finetune\MSD\Task06_Lung-001 + Task06_Lung
JSON_LIST: dataset_withVal_autogen.json
PRETRAIN_CKPT: D:\finetune\ar-ssl4m\ar-ssl4m\checkpoints\0\0.pth
SAVE_BASE: D:\finetune\msd_seg_ssl_runs


In [6]:
from monai.data import load_decathlon_datalist, load_decathlon_properties


def summarize_msd_task(task_dir: Path, json_filename: str) -> bool:
    task_dir = Path(task_dir).resolve()
    jp = task_dir / json_filename
    print("\n===== MSD 目录检查 =====")
    print("路径:", task_dir)
    if not jp.is_file():
        print(f"❌ 缺少 `{json_filename}`。")
        cands = [p.name for p in task_dir.glob("*.json")]
        print("当前目录下的 .json:", cands or "（无）")
        return False

    listing = sorted([p.name for p in task_dir.iterdir()])
    print("条目（节选）:", listing[:25], "…" if len(listing) > 25 else "")

    props_keys = ["name", "modality", "labels", "numTraining", "numValidation"]
    try:
        prop = load_decathlon_properties(str(jp), props_keys)
    except Exception as e:
        print("❌ load_decathlon_properties 失败:", e)
        return False

    print("dataset 名称:", prop.get("name"))
    print("modality:", prop.get("modality"))
    print("labels (类别字典):", prop.get("labels"))
    print("numTraining / numValidation:", prop.get("numTraining"), "/", prop.get("numValidation"))

    ok = True
    for split_key in ("training", "validation"):
        try:
            rows = load_decathlon_datalist(str(jp), True, split_key, base_dir=str(task_dir))
        except Exception as e:
            print(f"⚠️ 无法读取 split `{split_key}`:", e)
            ok = False
            continue
        print(f"[{split_key}] 条数: {len(rows)}")
        if not rows:
            print("⚠️ 列表为空")
            ok = False
            continue
        ex = rows[0]
        pi = Path(ex["image"]) if Path(ex["image"]).is_absolute() else task_dir / ex["image"]
        pl = Path(ex["label"]) if Path(ex["label"]).is_absolute() else task_dir / ex["label"]
        print("  首条 image:", pi, "exists" if pi.is_file() else "❌ MISSING")
        print("  首条 label:", pl, "exists" if pl.is_file() else "❌ MISSING")
        ok = ok and pi.is_file() and pl.is_file()
    print("========================================\n")
    return ok


_chk_ok = summarize_msd_task(MSD_TASK_DIR, JSON_LIST)
assert _chk_ok, "请先修正数据集路径或 JSON_LIST，再继续训练单元。"

c:\ProgramData\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



===== MSD 目录检查 =====
路径: D:\finetune\MSD\Task06_Lung-001\Task06_Lung
条目（节选）: ['._dataset.json', '._imagesTr', '._imagesTs', '._labelsTr', 'dataset.json', 'dataset_withVal_autogen.json', 'imagesTr', 'imagesTs', 'labelsTr'] 
dataset 名称: Lung
modality: {'0': 'CT'}
labels (类别字典): {'0': 'background', '1': 'cancer'}
numTraining / numValidation: 51 / 12
[training] 条数: 51
  首条 image: D:\finetune\MSD\Task06_Lung-001\Task06_Lung\imagesTr\lung_053.nii.gz exists
  首条 label: D:\finetune\MSD\Task06_Lung-001\Task06_Lung\labelsTr\lung_053.nii.gz exists
[validation] 条数: 12
  首条 image: D:\finetune\MSD\Task06_Lung-001\Task06_Lung\imagesTr\lung_025.nii.gz exists
  首条 label: D:\finetune\MSD\Task06_Lung-001\Task06_Lung\labelsTr\lung_025.nii.gz exists



In [1]:
%pip uninstall -y torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -y torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\ProgramData\miniconda3\python.exe -m pip install [options] <requirement specifier> [package-index-options] ...
  c:\ProgramData\miniconda3\python.exe -m pip install [options] -r <requirements file> [package-index-options] ...
  c:\ProgramData\miniconda3\python.exe -m pip install [options] [-e] <vcs project url> ...
  c:\ProgramData\miniconda3\python.exe -m pip install [options] [-e] <local project path> ...
  c:\ProgramData\miniconda3\python.exe -m pip install [options] <archive url/path> ...

no such option: -y


In [1]:
import tempfile
import torch
from pathlib import Path

_tmp_ckpt_root = tempfile.mkdtemp(prefix="seg_ssl_ckpt_")
_wrapped = Path(_tmp_ckpt_root) / "pretrain_segmentation.pth"

if "PRETRAIN_CKPT" not in globals():
    raise NameError("请先运行「配置路径」单元，定义 PRETRAIN_CKPT。")
if not Path(PRETRAIN_CKPT).is_file():
    raise FileNotFoundError(f"权重文件不存在: {PRETRAIN_CKPT}")

# PyTorch 2.6+ 默认可启用 weights_only；旧式 pickle .pth 建议 weights_only=False。
try:
    try:
        raw_blob = torch.load(PRETRAIN_CKPT, map_location="cpu", weights_only=False)
    except TypeError:
        raw_blob = torch.load(PRETRAIN_CKPT, map_location="cpu")
except ModuleNotFoundError as exc:
    msg = (
        f"{exc}\n"
        "常见于 PyTorch 混装/安装不完整（Roaming Python 与 conda 混用）。请在本内核解释器下：\n"
        "  python -m pip uninstall -y torch torchvision torchaudio\n"
        "再按笔记本「安装 PyTorch（cu130）」单元强制重装。"
    )
    raise RuntimeError(msg) from exc
if isinstance(raw_blob, dict) and "state_dict" in raw_blob and isinstance(raw_blob["state_dict"], dict):
    torch.save(raw_blob, _wrapped)
elif isinstance(raw_blob, dict):
    torch.save({"state_dict": raw_blob}, _wrapped)
else:
    raise TypeError("不支持的 checkpoint 类型")

print("供 segmentation/build.py 读取的封装文件:", _wrapped)
PRETRAIN_FOR_SEG = _wrapped

NameError: 请先运行「配置路径」单元，定义 PRETRAIN_CKPT。

In [ ]:
import os
import sys

import torch


def _cuda_diag_lines() -> list[str]:
    try:
        n = torch.cuda.device_count()
    except Exception as e:
        n = f"(error: {e})"
    return [
        f"  torch.__version__ = {torch.__version__!r}",
        f"  torch.version.cuda = {torch.version.cuda!r}",
        f"  torch.cuda.is_available() = {torch.cuda.is_available()}",
        f"  torch.cuda.device_count() = {n}",
        f"  sys.executable = {sys.executable!r}",
    ]


if not torch.cuda.is_available():
    msg = (
        "`main.py` 训练路径依赖 CUDA，但当前 Jupyter 内核里 torch.cuda.is_available() 为 False。\n\n"
        "【诊断】\n"
        + "\n".join(_cuda_diag_lines())
        + "\n\n"
        "【常见原因】\n"
        "  • 本环境是 CPU 版 PyTorch，或 GPU 轮子与驱动/CUDA 不匹配。\n"
        "  • 笔记本选用的内核不是你安装 GPU torch 的那个 conda/venv（混用 `--user` pip 时常出现）。\n"
        "  • 远程/容器无 GPU，或未正确透传。\n\n"
        "【建议】用「与第一个 PyTorch 安装单元相同的解释器」在终端运行：\n"
        '  python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"\n'
        "确认第三项为 True 后，在 Cursor / VS Code 里把 Jupyter 内核切到该解释器，再从上到下重跑笔记本。"
    )
    raise RuntimeError(msg)

SEG_DIR.mkdir(parents=True, exist_ok=True)
SAVE_BASE.mkdir(parents=True, exist_ok=True)

old_argv = sys.argv.copy()
old_cwd = os.getcwd()
# 使 downstream/segmentation 下相对导入与资源路径可用
os.chdir(str(SEG_DIR))
if str(SEG_DIR) not in sys.path:
    sys.path.insert(0, str(SEG_DIR))

_argv_list = [
    "main.py",
    "--MSD_data_base",
    str(MSD_DATA_BASE),
    "--save_base",
    str(SAVE_BASE),
    "--task_name",
    TASK_NAME,
    "--json_list",
    JSON_LIST,
    "--pretrain_path",
    str(PRETRAIN_FOR_SEG),
    "--workers",
    "2",
]

# 不推荐 noamp —— 与原脚本 AMP 对齐；若在笔记本里 OOM，可在此处加上 "--noamp"

if DEMO_QUICK:
    _argv_list += [
        "--max_epochs",
        "2",
        "--val_every",
        "1",
        "--use_normal_dataset",
    ]

sys.argv = _argv_list

import importlib.util

_spec = importlib.util.spec_from_file_location("seg_ssl_main", SEG_DIR / "main.py")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)

try:
    print("Starting segmentation main() with argv:")
    print(" ", " ".join(_argv_list[1:]))
    _mod.main()
finally:
    sys.argv = old_argv
    os.chdir(old_cwd)

print("完成。日志与模型在 SAVE_BASE/task_name/exp 下，参见 main.py 生成的 log.txt。")

RuntimeError: 当前 segmentation 流水线固定使用 CUDA；请改用有 GPU 的环境。